# Choosing `space_time_ratio` for TAM3C2

TAM3C2 aggregates point-cloud neighbourhoods **jointly in space and time** before computing the M3C2 distance. The temporal half of that aggregation is controlled by a single unitless parameter — `space_time_ratio` ($r_{st}$).

This notebook explains the parameter, then runs an automated sweep to pick a value that *maximises the temporal aggregation without polluting the M3C2 spread with real surface change*.  
Larger $r_{st}$ ⇒ more temporal averaging ⇒ smaller LoD; but if you push it too far, real evolution leaks into the spread. We sweep candidate ratios and pick the largest one whose spread inflation stays under a tolerance.

## 1. Background — why a space-time ratio?

Inside one corepoint neighbourhood, every point $i$ has

* a spatial offset $d^{(s)}_i = \|x_i - x_{cp}\|$ measured in **metres**, and
* a temporal offset $d^{(t)}_i = |t_i - t_{ref}|$ measured in **seconds**.

These two have **different units**, so we first normalise them by their respective window sizes:

$$
u^{(s)}_i = \frac{d^{(s)}_i}{r_{\text{spatial}}}, \qquad u^{(t)}_i = \frac{d^{(t)}_i}{W}.
$$

Here $r_{\text{spatial}}$ is the cylinder radius and $W = |t_{tgt}-t_{ref}|\cdot \text{max\_window\_ratio}$ is the temporal half-window.

Even after that normalisation we still have a choice: how much should we *trust* a one-window-away neighbour in time versus a one-window-away neighbour in space? That is precisely a **kriging-style anisotropy** decision. We parameterise it with a single ratio

$$
r_{st} = \frac{(\text{trust radius in time})}{(\text{trust radius in space})}
$$

and apply an isotropic Gaussian to the *rescaled* coordinates $(u^{(s)}, u^{(t)}/r_{st})$. Concretely the weight on a neighbour is

$$
w_i \;=\; \underbrace{\exp\!\Bigl(-\tfrac{1}{2\sigma^{2}}\,u^{(s)2}_i\Bigr)}_{w^{(s)}_i}
         \;\cdot\;
         \underbrace{\exp\!\Bigl(-\tfrac{1}{2\sigma^{2}}\bigl(u^{(t)}_i/r_{st}\bigr)^{2}\Bigr)}_{w^{(t)}_i}
$$

with $\sigma$ = `sigma_ratio`.

Interpretation of $r_{st}$ (with `sigma_ratio = 1` for orientation):

| $r_{st}$ | Meaning |
|---|---|
| **= 1** | symmetric: a 1-window time offset is penalised the same as a 1-radius space offset |
| **> 1** | trust time *more*: temporal weight decays slower → more epochs aggregate → smoother / lower LoD |
| **< 1** | trust space *more*: temporal weight decays faster → fewer epochs aggregate → closer to single-epoch M3C2 |

**Hard cap.** Whatever $r_{st}$ you pick, the hard cutoff `max_window_ratio` (and the `required_points` check) still applies — $r_{st}$ only reshapes the *soft* decay inside that window.


## 2. How do we pick $r_{st}$? — spread-inflation criterion

Pick a corepoint and a side (ref or tgt). The M3C2 along-normal projections of its aggregated points have variance

$$
\mathrm{Var}(p) \;=\; \underbrace{\sigma_{\text{geom}}^{2}}_{\text{intra-epoch roughness}}
                   \;+\; \underbrace{\mathrm{Var}_{e}\!\bigl(\bar{p}_{e}\bigr)}_{\text{inter-epoch drift}}
$$

(law of total variance, with $e$ indexing epochs). The first term is what we *want* in M3C2 — it is the geometric noise the LoD formula assumes. The second term is *contamination*: any real surface evolution that happens within the aggregation window mixes into the spread and inflates LoD spuriously.

**Stable corepoints are the canary.** On a corepoint that does *not* change over the whole time series, the second term has nothing to push around — *unless* we widen the temporal window enough that microscopic drift creeps in. So we watch the spread on stable corepoints as a function of $r_{st}$.

Define

$$
\bar s(r_{st}) \;=\; \frac{1}{|\mathcal{S}|}\sum_{cp\in\mathcal{S}}\frac{1}{T}\sum_{k=1}^{T}\tfrac{1}{2}\bigl(s_{\text{ref}}(cp,k;r_{st}) + s_{\text{tgt}}(cp,k;r_{st})\bigr)
$$

where $\mathcal{S}$ is the stable-corepoint set, $T$ is the number of targets, and $s_{\text{ref}}, s_{\text{tgt}}$ are the weighted standard deviations the M3C2 calculation already emits in the `uncertainties` array.

The **spread inflation ratio** is

$$
\rho(r_{st}) \;=\; \frac{\bar s(r_{st})}{\bar s(r_{st}^{\min})}
$$

where $r_{st}^{\min}$ is the smallest ratio in our sweep — it serves as the geometric-noise baseline (it does the least temporal smoothing of the candidates we consider).

**Selection rule.** Given a tolerance $\tau$ (default $0.1 = 10\%$):

$$
r_{st}^{\star} \;=\; \max\bigl\{\, r_{st}\in\mathcal{C} \,:\, \rho(r_{st}) \le 1 + \tau \,\bigr\}
$$

i.e. **the largest candidate ratio whose stable spread is at most 10 % above the baseline**. This buys the most LoD reduction we can get for an essentially-unchanged spread.

---
## 3. Algorithm — what `estimate_space_time_ratio` does step by step

Inputs and their roles:

| Argument | Meaning |
|---|---|
| `candidate_ratios` | the grid we sweep; default `(0.5, 1.0, 2.0, 4.0)` |
| `tolerance` | the $\tau$ in $\rho \le 1+\tau$; default `0.1` |
| `stable_threshold` | a corepoint is *stable* if $\max_t |d(cp,t)| < $ this value (in metres). Use `obc_height_threshold`. |
| `tam_kwargs` | every other TAM3C2 / M3C2 parameter (cyl_radius, max_distance, ...). Whatever you would pass to the constructor in the main pipeline. |

Procedure:

1. For every $r_{st}\in\mathcal{C}$, build a fresh `TAM3C2(space_time_ratio=r_st, **tam_kwargs)` and call `calculate_distances(ref, tgt)` for every target. Collect the $(n_{cp}\times n_{tgt})$ matrices of distances, spreads and LoDs.
2. Use the **smallest** $r_{st}$ as baseline. Mark stable corepoints with $\max_t |d| < $ `stable_threshold`.
3. Compute $\bar s(r_{st})$ on the stable set for every ratio, then $\rho(r_{st})$.
4. Return the largest ratio satisfying $\rho \le 1+\tau$ (or, if none qualifies, the ratio with the smallest $\rho$ together with `constraint_satisfied=False`).

The returned `dict` also exposes the full per-ratio report so you can plot and report any of:

- $\rho(r_{st})$ — the spread inflation curve;
- mean LoD$_{95}$ — should fall as $r_{st}$ grows;
- mean effective sample count $N_{\text{eff}}$ — should rise as $r_{st}$ grows.

## 4. Setup

We reuse the same data and parameters as the main `test_S1_uls_tam3c2.ipynb` so the chosen $r_{st}$ transfers directly.

In [ ]:
from datetime import datetime
import os, numpy as np, matplotlib.pyplot as plt
import py4dgeo
from py4dgeo import (
    TAM3C2, Weighting,
    read_epochs_from_folder, extract_reference_and_others, sample_corepoints,
    sweep_space_time_ratio, estimate_space_time_ratio,
)

In [ ]:
data_path = r'C:\rsa\research_proj\time_aware_M3C2\test\salt_marsh\Rennes_database_COSMA_downsampled'

reference_timestamp = datetime(2010, 10, 4)
max_epochs_after_reference = 120

# Corepoint sampling
corepoint_voxel_size = 1.5

# TAM3C2 / M3C2 parameters (same as the main notebook)
normal_radii      = [0.5, 1.0, 1.5]
max_window_ratio  = [0.2, 0.3]
required_points   = 10
cyl_radius        = 1.0
max_distance      = 5.0
registration_error = 0.02
sigma_ratio       = 1.0
weighting         = Weighting.GAUSSIAN

# Stable-corepoint threshold (use the same value as obc_height_threshold)
stable_threshold = 0.1

# Ratio sweep settings
candidate_ratios = (0.5, 1.0, 2.0, 4.0)
tolerance        = 0.1  # 10 % inflation budget

# To keep the sweep fast we evaluate the criterion on a subset of targets
n_sweep_targets = 30

In [ ]:
epochs = read_epochs_from_folder(data_path)
if max_epochs_after_reference is not None:
    sorted_eps = sorted(epochs, key=lambda e: e.timestamp)
    ref_idx = next(i for i, e in enumerate(sorted_eps) if e.timestamp == reference_timestamp)
    epochs = sorted_eps[: ref_idx + 1 + max_epochs_after_reference]

reference_epoch, other_epochs = extract_reference_and_others(epochs, reference_timestamp)
corepoints = sample_corepoints(reference_epoch, method='voxel', voxel_size=corepoint_voxel_size)
print(f"epochs: {len(epochs)}  |  reference: {reference_epoch.timestamp}  |  corepoints: {len(corepoints):,}")

# Evenly spaced subset of targets for the sweep
idx = np.linspace(0, len(other_epochs) - 1, num=min(n_sweep_targets, len(other_epochs)), dtype=int)
target_subset = [other_epochs[i] for i in idx]
print(f"sweep targets: {len(target_subset)}")

## 5. Run the sweep + selection

Everything that would normally be passed to `TAM3C2(...)` goes into `tam_kwargs` (everything except `space_time_ratio`, `epochs_timeseries` and `corepoints`, which `estimate_space_time_ratio` controls itself).

In [ ]:
tam_kwargs = dict(
    max_window_ratio=max_window_ratio,
    normal_radii=normal_radii,
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

result = estimate_space_time_ratio(
    epochs_timeseries=epochs,
    reference_epoch=reference_epoch,
    target_epochs=target_subset,
    corepoints=corepoints,
    tam_kwargs=tam_kwargs,
    candidate_ratios=candidate_ratios,
    tolerance=tolerance,
    stable_threshold=stable_threshold,
)

print(f"best space_time_ratio = {result['best_ratio']}")
print(f"constraint satisfied  = {result['constraint_satisfied']}")
print(f"baseline ratio        = {result['baseline_ratio']}  (used to define rho = 1)")
print(f"#stable corepoints    = {int(result['stable_mask'].sum())} / {len(corepoints)}")

In [ ]:
# Per-ratio report
print(f"{'r_st':>6}  {'rho':>8}  {'mean LoD95 [m]':>16}  {'mean N_eff':>12}")
for r in result['report']:
    marker = '  <-- best' if r['space_time_ratio'] == result['best_ratio'] else ''
    print(f"{r['space_time_ratio']:>6.2f}  {r['rho']:>8.4f}  {r['mean_lod95']:>16.4f}  {r['mean_num_samples']:>12.1f}{marker}")

## 6. Diagnostic plots

Three curves tell the whole story:

1. **$\rho(r_{st})$** — spread inflation on stable corepoints. We want this to stay below $1+\tau$.
2. **mean LoD$_{95}$** — should drop as $r_{st}$ grows (the *benefit* of aggregation).
3. **mean $N_{\text{eff}}$** — should rise as $r_{st}$ grows (the *evidence* that we are actually aggregating).

In [ ]:
ratios   = np.array([r['space_time_ratio']  for r in result['report']])
rhos     = np.array([r['rho']               for r in result['report']])
lods     = np.array([r['mean_lod95']        for r in result['report']])
neffs    = np.array([r['mean_num_samples']  for r in result['report']])
best_r   = result['best_ratio']

fig, axs = plt.subplots(1, 3, figsize=(15, 4))

axs[0].plot(ratios, rhos, 'o-')
axs[0].axhline(1.0 + tolerance, color='r', ls='--', label=f'1 + tolerance = {1+tolerance:.2f}')
axs[0].axvline(best_r,          color='g', ls='--', label=f'selected r_st = {best_r}')
axs[0].set_xscale('log')
axs[0].set_xlabel('space_time_ratio  (log)')
axs[0].set_ylabel(r'spread inflation $\rho$')
axs[0].set_title('Stable-corepoint spread inflation')
axs[0].grid(alpha=0.3)
axs[0].legend()

axs[1].plot(ratios, lods, 'o-')
axs[1].axvline(best_r, color='g', ls='--')
axs[1].set_xscale('log')
axs[1].set_xlabel('space_time_ratio  (log)')
axs[1].set_ylabel('mean LoD95 [m]')
axs[1].set_title('Detection threshold')
axs[1].grid(alpha=0.3)

axs[2].plot(ratios, neffs, 'o-')
axs[2].axvline(best_r, color='g', ls='--')
axs[2].set_xscale('log')
axs[2].set_xlabel('space_time_ratio  (log)')
axs[2].set_ylabel(r'mean $N_{eff}$')
axs[2].set_title('Effective sample count')
axs[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Spatial distribution of the stable corepoints used in rho(r_st)
stable = result['stable_mask']
plt.figure(figsize=(9, 5))
plt.scatter(corepoints[~stable, 0], corepoints[~stable, 1], s=5, c='lightgrey', label='changing')
plt.scatter(corepoints[ stable, 0], corepoints[ stable, 1], s=8, c='tab:blue',  label='stable (baseline)')
plt.gca().set_aspect('equal')
plt.xlabel('X [m]'); plt.ylabel('Y [m]')
plt.title(f'Stable corepoints ({int(stable.sum())}/{len(corepoints)}) — used for rho')
plt.legend(); plt.tight_layout(); plt.show()

## 7. Use the chosen value in the main pipeline

Set the result in the main notebook's configuration cell:

```python
space_time_ratio = result['best_ratio']  # e.g. 2.0
```

and rebuild `TAM3C2(..., space_time_ratio=space_time_ratio, ...)`.

### What to report (PPT-ready bullet list)

* **Parameter.** $r_{st}$ = anisotropy ratio between the *temporal* and *spatial* trust radii of TAM3C2's Gaussian kernel.
* **Weight model.** $w_i = \exp\bigl(-\tfrac{1}{2\sigma^2}u^{(s)2}_i\bigr)\,\exp\bigl(-\tfrac{1}{2\sigma^2}(u^{(t)}_i/r_{st})^2\bigr)$ with $u = d/\text{window}$.
* **Selection criterion.** Spread inflation on stable corepoints,
  $\rho(r_{st}) = \bar s(r_{st})/\bar s(r_{st}^{\min})$, must satisfy $\rho \le 1+\tau$.
* **Decision.** Pick the largest candidate ratio that satisfies the bound (most LoD reduction, ≤ τ pollution).
* **Defaults used here.** candidates `(0.5, 1.0, 2.0, 4.0)`, $\tau = 0.1$, `stable_threshold` = `obc_height_threshold` = 0.05 m.
* **Result.** Quote `best_ratio`, the $\rho$-curve, the LoD reduction from `r_st=baseline` to `r_st=best`, and the number of stable corepoints used.